# Análisis Exploratorio de Datos: Elecciones al Senado de Colombia 2018
### Notebook pedagógico — Curso de Analítica de Datos y Machine Learning en Ciencias Políticas
**Universidad de Antioquia** — Facultad de Derecho y Ciencias Políticas — 2026-1

---

#### ¿Qué es un EDA y por qué lo hacemos?

Un **Análisis Exploratorio de Datos (EDA)** es el proceso sistemático de examinar un dataset antes de formular modelos o realizar inferencias. El concepto fue introducido por el estadístico John Tukey en 1977 y sigue siendo la piedra angular de cualquier proyecto serio de ciencia de datos.

Piensa en el EDA como la exploración de un territorio desconocido: necesitamos recorrerlo, observar su geografía y sus caminos antes de trazar un mapa. En nuestro caso, el «territorio» son los **resultados electorales al Senado de la República de Colombia en las elecciones de 2018**, provenientes del CEDAE de la Registraduría Nacional del Estado Civil.

**¿Qué contiene el dataset?** Los votos obtenidos por cada partido y cada candidato en cada municipio del país, tanto para la **circunscripción Nacional** como para la **Indígena**.

**Preguntas que guiarán nuestra exploración:**
- ¿Cómo se distribuye el voto entre partidos y candidatos?
- ¿Hay unos pocos candidatos que concentran la mayoría de los votos, o la competencia es pareja?
- ¿Qué diferencias existen entre la circunscripción Nacional y la Indígena?
- ¿La competencia electoral varía significativamente entre departamentos?
- ¿Qué variables podrían predecir si un candidato obtiene curul?

**Estructura del notebook:** Seguiremos un roadmap de 23 pasos organizados en tres niveles:
- 🟢 **Básico** (Pasos 1–6): Auditoría estructural y panorama inicial
- 🟡 **Intermedio** (Pasos 7–14): Limpieza, interacciones y contexto
- 🔴 **Avanzado** (Pasos 15–23): Estructura latente y preparación para modelado

> 💡 **Ojito:** Cada paso comienza con una explicación de *qué* se va a hacer y *por qué*. Lee las celdas de texto antes de ejecutar el código — entender la lógica es tan importante como ejecutar las instrucciones.

In [16]:
# CONFIGURACIÓN GLOBAL E IMPORTACIÓN DE LIBRERÍAS

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
from scipy.stats import mannwhitneyu, chi2_contingency, kruskal
import missingno as msno
import warnings

# Configuración de visualización
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 12
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
sns.set_style('whitegrid')

# Paleta de colores personalizada para todo el notebook
COLORES = {
    'principal': '#2c3e50',
    'secundario': '#7f8c8d',
    'acento': '#e67e22',
    'positivo': '#27ae60',
    'negativo': '#c0392b',
    'claro': '#ecf0f1',
}
PALETA_PARTIDOS = sns.color_palette('Set2', 20)
PALETA_SEQ = sns.color_palette('Blues_r', 10)

# Configuración de pandas
pd.set_option('display.max_columns', 20)
pd.set_option('display.max_rows', 60)
pd.set_option('display.float_format', '{:.2f}'.format)
pd.set_option('display.max_colwidth', 60)

# Semilla para reproducibilidad
np.random.seed(42)

# Suprimir advertencias menores
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

print("Librerías cargadas y configuración global aplicada.")

Librerías cargadas y configuración global aplicada.


---
# 🟢 Nivel Básico — Auditoría Estructural y Panorama Inicial

**Objetivo:** ¿Qué tenemos entre manos, cuál es la forma de los datos y en qué estado de calidad se encuentran?

En este nivel vamos a conocer el dataset como si fuera un nuevo vecindario: recorrer sus calles, contar sus casas, verificar que las direcciones sean correctas y tomar nota de lo que se ve a primera vista.

## Paso 1 · Carga y primer contacto
🟢 Básico

Antes de cualquier análisis, necesitamos **cargar los datos y mirarlos**. Así como un politólogo empieza su investigación leyendo las fuentes primarias, un analista de datos empieza leyendo sus datos.

Vamos a:
1. Importar el archivo CSV con `pandas`
2. Ver las primeras y últimas filas (`.head()`, `.tail()`)
3. Tomar una muestra aleatoria (`.sample()`)
4. Verificar las dimensiones: ¿cuántas filas y columnas tiene?

> **Pregunta guía:** ¿Cuántos registros tiene el dataset y qué representa cada fila? ¿Es un registro por candidato, por partido, por municipio, o una combinación?

In [4]:
# Cargar el dataset desde el archivo CSV
df = pd.read_csv('senado_2018.csv')

# Dimensiones del dataset
print(f"Dimensiones del dataset: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"   Esto significa {df.shape[0]:,} registros, cada uno con {df.shape[1]} variables.\n")

# Primeras filas: ver la estructura general
print("═" * 80)
print("PRIMERAS 5 FILAS")
print("═" * 80)
df.head()

Dimensiones del dataset: 318,620 filas × 16 columnas
   Esto significa 318,620 registros, cada uno con 16 variables.

════════════════════════════════════════════════════════════════════════════════
PRIMERAS 5 FILAS
════════════════════════════════════════════════════════════════════════════════


,id_electoral,ano,tipo_eleccion,fecha_eleccion,coddpto,departamento,codmpio,municipio,circunscripcion,codigo_partido,codigo_lista,primer_apellido,segundo_apellido,nombres,votos,curules
0,220180001,2018,Senado,Marzo 11,5,ANTIOQUIA,5001,MEDELLIN,Nacional,20000036.00,0,NaN,NaN,PARTIDO POLITICO MIRA,3525,0
1,220180001,2018,Senado,Marzo 11,5,ANTIOQUIA,5001,MEDELLIN,Nacional,20130001.00,0,NaN,NaN,PARTIDO CENTRO DEMOCRATICO,62056,0
2,220180001,2018,Senado,Marzo 11,5,ANTIOQUIA,5001,MEDELLIN,Nacional,20090002.00,0,NaN,NaN,PARTIDO ALIANZA VERDE,11607,0
3,220180001,2018,Senado,Marzo 11,5,ANTIOQUIA,5001,MEDELLIN,Nacional,20030001.00,0,NaN,NaN,PARTIDO CAMBIO RADICAL,4905,0
4,220180001,2018,Senado,Marzo 11,5,ANTIOQUIA,5001,MEDELLIN,Nacional,20170001.00,0,NaN,NaN,GSC COLOMBIA JUSTA LIBRES,10977,0


In [5]:
# Últimas 5 filas: verificar que el archivo se cargó completo
print("═" * 80)
print("ÚLTIMAS 5 FILAS")
print("═" * 80)
df.tail()

════════════════════════════════════════════════════════════════════════════════
ÚLTIMAS 5 FILAS
════════════════════════════════════════════════════════════════════════════════


,id_electoral,ano,tipo_eleccion,fecha_eleccion,coddpto,departamento,codmpio,municipio,circunscripcion,codigo_partido,codigo_lista,primer_apellido,segundo_apellido,nombres,votos,curules
318615,220180002,2018,Senado,Marzo 11,99,VICHADA,99773,CUMARIBO,Indígena,19910006.00,203,QUIGUA,IZQUIERDO,ATI SEYGUNDIBA,4,0
318616,220180002,2018,Senado,Marzo 11,99,VICHADA,99773,CUMARIBO,Indígena,NaN,996,VOTOS EN BLANCO INDIGENAS,NaN,NaN,108,0
318617,220180001,2018,Senado,Marzo 11,99,VICHADA,99773,CUMARIBO,Nacional,NaN,997,TARJETAS NO MARCADAS,NaN,NaN,2808,0
318618,220180001,2018,Senado,Marzo 11,99,VICHADA,99773,CUMARIBO,Nacional,NaN,998,VOTOS NULOS,NaN,NaN,244,0
318619,220180001,2018,Senado,Marzo 11,99,VICHADA,99773,CUMARIBO,Nacional,NaN,999,VOTOS EN BLANCO,NaN,NaN,201,0


In [8]:
# Muestra aleatoria: obtener una visión diversa
print("═" * 80)
print("MUESTRA ALEATORIA DE 10 FILAS")
print("═" * 80)
df.sample(5)

════════════════════════════════════════════════════════════════════════════════
MUESTRA ALEATORIA DE 10 FILAS
════════════════════════════════════════════════════════════════════════════════


,id_electoral,ano,tipo_eleccion,fecha_eleccion,coddpto,departamento,codmpio,municipio,circunscripcion,codigo_partido,codigo_lista,primer_apellido,segundo_apellido,nombres,votos,curules
237752,220180001,2018,Senado,Marzo 11,66,RISARALDA,66001,PEREIRA,Nacional,18480001.00,96,CORRALES,CANO,JESUS,3,0
29338,220180001,2018,Senado,Marzo 11,5,ANTIOQUIA,5665,SAN PEDRO DE URABA,Nacional,20030001.00,69,MENDOZA,MENDOZA,MARIO RAMON,1,0
110436,220180001,2018,Senado,Marzo 11,19,CAUCA,19533,PIAMONTE,Nacional,20030001.00,24,VIAFARA,NaN,BERTHALY,1,0
18158,220180001,2018,Senado,Marzo 11,5,ANTIOQUIA,5360,ITAGUI,Nacional,18490002.00,39,ROJAS,ARANGO,CONRADO DE JESUS,4,0
307663,220180001,2018,Senado,Marzo 11,85,CASANARE,85400,TAMARA,Nacional,20000036.00,0,NaN,NaN,PARTIDO POLITICO MIRA,5,0


In [ ]:
# Nombres de las columnas
print("Columnas del dataset:")
for i, col in enumerate(df.columns, 1):
    print(f"   {i:2d}. {col}")

### 🔍 Interpretación — Paso 1

El dataset contiene más de **300,000 registros** y **16 columnas**. Cada fila representa una combinación única de **municipio × partido o candidato**. Esto quiere decir que un mismo candidato (por ejemplo, Álvaro Uribe) aparece **una vez por cada municipio** del país donde está registrada la votación.

Hay dos tipos de filas:
- **Filas de partido** (`codigo_lista = 0`): consolidan los votos de lista cerrada del partido en ese municipio. En la columna `nombres` aparece el nombre del partido.
- **Filas de candidato** (`codigo_lista ≥ 1`): votos individuales de cada candidato en ese municipio. La columna `nombres` contiene el primer nombre del candidato, y `primer_apellido` / `segundo_apellido` completan la identificación.

Esta estructura «granular» es muy rica porque permite análisis a múltiples niveles (candidato, partido, municipio, departamento), pero implica que debemos tener **cuidado de no mezclar los dos tipos de filas** al calcular estadísticas.

## Paso 2 · Estructura y tipos de datos
🟢 Básico

Cada columna de un dataset tiene un **tipo de dato** asignado automáticamente al cargar el archivo. Un error muy frecuente es que el sistema asigne un tipo incorrecto.

> **Analogía política:** El código DANE de Antioquia es `5` y el de Bogotá es `11`. ¿Tiene sentido calcular el *promedio* de esos códigos? ¡Por supuesto que no! Son **etiquetas categóricas** disfrazadas de números. Sumarlos o promediarlos produciría un resultado sin sentido.

> **Pregunta guía:** ¿Los tipos de datos asignados automáticamente son correctos? ¿Hay variables categóricas mal clasificadas como numéricas?

In [ ]:
# Información general del dataset: tipos, nulos y memoria
print("═" * 80)
print("ESTRUCTURA DEL DATASET (.info())")
print("═" * 80)
df.info()

In [ ]:
# Tipos de datos asignados automáticamente
print("═" * 80)
print("TIPOS DE DATOS POR COLUMNA")
print("═" * 80)
print(df.dtypes)
print(f"\nOJITO: Observa que 'coddpto' y 'codmpio' fueron cargados como int64 (entero).")
print("   Estos son CÓDIGOS DANE, no cantidades. Debemos convertirlos a string/categórico.")
print(f"\n   Ejemplo: coddpto tiene valores como {sorted(df['coddpto'].unique())[:5]}...")
print(f"   codmpio tiene valores como {sorted(df['codmpio'].unique())[:5]}...")

In [ ]:
# ── CORRECCIÓN DE TIPOS ──
# Convertir códigos numéricos a string porque son identificadores, no cantidades
df['coddpto'] = df['coddpto'].astype(str)
df['codmpio'] = df['codmpio'].astype(str)
df['codigo_partido'] = df['codigo_partido'].astype(str)
df['id_electoral'] = df['id_electoral'].astype(str)

# Verificar la corrección
print("Tipos corregidos:")
print(df[['id_electoral', 'coddpto', 'codmpio', 'codigo_partido']].dtypes)
print(f"\n   Ahora 'coddpto' es tipo: {df['coddpto'].dtype}")
print(f"   Ejemplo: {df['coddpto'].unique()[:5]}")

### 🔍 Interpretación — Paso 2

Detectamos un problema clásico: los **códigos DANE** (`coddpto`, `codmpio`) fueron cargados como enteros. Lo mismo ocurrió con `codigo_partido` e `id_electoral`.

Estos campos no son cantidades sino **identificadores categóricos**. Si los dejáramos como enteros:
- Un gráfico podría ordenar los departamentos «numéricamente» en lugar de por nombre.
- Un modelo de ML los trataría como variable continua y asumiría que departamento «15» está «más lejos» de «5» que de «8».

> **Lección general:** Siempre verifica los tipos de datos al cargar un dataset. Los sistemas automáticos no conocen el contexto — esa es tu responsabilidad como analista.

## Paso 3 · Diagnóstico de valores faltantes
🟢 Básico

Los **valores faltantes** (nulos, `NaN`) son una realidad en prácticamente cualquier dataset. Pero no todos los faltantes son iguales. En estadística distinguimos tres tipos:

| Tipo | Significado | Ejemplo |
|------|-------------|---------|
| **MCAR** (*Missing Completely At Random*) | El dato falta por razones completamente aleatorias | Se cayó una hoja y se perdieron datos al azar |
| **MAR** (*Missing At Random*) | Falta dependiendo de *otra* variable observable | Municipios pequeños reportan menos datos |
| **MNAR** (*Missing Not At Random*) | Falta por razones que dependen del *propio dato* faltante | El dato simplemente no debería existir para esa fila |

> **Pregunta guía:** ¿Hay valores nulos en el dataset? ¿Son aleatorios o responden a un patrón estructural?

In [ ]:
# ── CONTEO DE NULOS ──
print("═" * 80)
print("VALORES FALTANTES POR COLUMNA")
print("═" * 80)
nulos = pd.DataFrame({
    'Nulos': df.isnull().sum(),
    'Porcentaje': (df.isnull().sum() / len(df) * 100).round(2)
})
nulos = nulos[nulos['Nulos'] > 0].sort_values('Porcentaje', ascending=False)

if len(nulos) > 0:
    print(nulos.to_string())
    print(f"\nTotal de celdas con datos faltantes: {df.isnull().sum().sum():,}")
else:
    print("No hay valores nulos en el dataset.")

# Verificar: ¿los nulos en primer_apellido coinciden con filas de partido?
nulos_apellido = df['primer_apellido'].isnull()
filas_partido = df['codigo_lista'] == 0
coincidencia = (nulos_apellido == filas_partido).mean() * 100
print(f"\nLos nulos en 'primer_apellido' coinciden con filas de partido (codigo_lista=0)")
print(f"   en el {coincidencia:.1f}% de los casos.")

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
msno.matrix(df.sample(1000, random_state=42), ax=ax, sparkline=False,
            fontsize=10, color=(0.17, 0.24, 0.31))
ax.set_title('Mapa de valores faltantes (muestra de 1,000 filas)',
             fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

### 🔍 Interpretación — Paso 3

Los valores nulos aparecen **exclusivamente** en las columnas `primer_apellido` y `segundo_apellido`, y coinciden casi perfectamente con las filas de partido (`codigo_lista = 0`).

Esto **no es un error de recolección** sino un diseño estructural del dataset:
- Las filas de **partido** no tienen persona asociada → `primer_apellido` y `segundo_apellido` están vacíos.
- Las filas de **candidato** (`codigo_lista ≥ 1`) siempre tienen nombre.

En la terminología de valores faltantes, esto es **MNAR** (*Missing Not At Random*): los datos faltan porque *no deberían existir*. No hay un nombre que «se perdió» — simplemente no aplica.

> **Decisión:** No vamos a imputar estos nulos. Son consecuencia de la estructura del dataset donde coexisten dos niveles (partido y candidato). Cuando sea necesario, trabajaremos con subconjuntos filtrados por `codigo_lista`.

## Paso 4 · Diagnóstico de duplicados y rangos plausibles
🟢 Básico

Este paso es la **primera línea de defensa** contra datos corruptos. Un solo duplicado puede distorsionar un promedio; un voto negativo puede sabotear una gráfica.

Vamos a verificar:
1. ¿Hay filas completamente duplicadas?
2. ¿Hay duplicados por **clave lógica** (`id_electoral` + `codigo_partido` + `codigo_lista`)?
3. ¿Los valores numéricos están en rangos razonables? (votos ≥ 0, curules ∈ {0,1}, etc.)

> **Pregunta guía:** ¿Podemos confiar en que cada fila es un registro único y que los valores son plausibles?

In [ ]:
# ── DIAGNÓSTICO DE DUPLICADOS ──
print("═" * 80)
print("DIAGNÓSTICO DE DUPLICADOS")
print("═" * 80)

# Duplicados exactos
dup_exactos = df.duplicated().sum()
print(f"Filas completamente duplicadas: {dup_exactos:,}")

# Duplicados por clave lógica
clave = ['id_electoral', 'codigo_partido', 'codigo_lista']
dup_clave = df.duplicated(subset=clave).sum()
print(f"Duplicados por clave lógica ({' + '.join(clave)}): {dup_clave:,}")

if dup_clave > 0:
    print("\nHay duplicados por clave lógica. Ejemplo:")
    duplicados = df[df.duplicated(subset=clave, keep=False)].sort_values(clave)
    print(duplicados.head(10))
else:
    print("\nNo hay duplicados por clave lógica — cada fila es un registro único.")

In [ ]:
# ── VERIFICACIÓN DE RANGOS PLAUSIBLES ──
print("═" * 80)
print("VERIFICACIÓN DE RANGOS PLAUSIBLES")
print("═" * 80)

# Votos negativos
votos_neg = (df['votos'] < 0).sum()
print(f"Votos negativos: {votos_neg}")
print(f"Rango de votos: [{df['votos'].min():,} — {df['votos'].max():,}]")

# Curules fuera de {0, 1}
curules_invalidas = df[~df['curules'].isin([0, 1])].shape[0]
print(f"\nFilas con curules distintas de 0 o 1: {curules_invalidas}")
print(f"Valores únicos de curules: {sorted(df['curules'].unique())}")

# Verificaciones adicionales
n_deptos = df['coddpto'].nunique()
print(f"\nDepartamentos únicos (códigos DANE): {n_deptos}")
print(f"Años presentes: {df['ano'].unique()}")
print(f"Circunscripciones: {df['circunscripcion'].unique()}")
print(f"Tipos de elección: {df['tipo_eleccion'].unique()}")

# ¿Hay candidatos con curules=1 y 0 votos totales a nivel nacional?
if len(df[df['codigo_lista'] > 0]) > 0:
    votos_nac = df[df['codigo_lista'] > 0].groupby(
        ['primer_apellido', 'segundo_apellido', 'nombres']
    )['votos'].sum()
    elegidos_sin_votos = votos_nac[votos_nac == 0]
    print(f"\nCandidatos con 0 votos totales en todo el país: {len(elegidos_sin_votos)}")

### 🔍 Interpretación — Paso 4

**Buenas noticias:**
- **No hay duplicados** por clave lógica — cada combinación municipio-partido-posición es única.
- **No hay votos negativos** ni curules con valores imposibles.
- Los datos corresponden exclusivamente al **año 2018** y a la elección de **Senado**.

Hay candidatos con **0 votos** en muchos municipios; esto no es un error. En el voto preferente colombiano, un candidato aparece en la tarjeta electoral de todos los municipios pero solo recibe votos en algunos. Los ceros son información legítima.

> **Lección:** En datos oficiales de la Registraduría la calidad suele ser buena. En datasets de menor calidad (encuestas, scraping, registros administrativos locales), este paso suele revelar sorpresas desagradables.

## Paso 5 · Estadística descriptiva univariada
🟢 Básico

La **estadística descriptiva** nos da un resumen numérico de cada variable. Las medidas clave son:
- **Tendencia central:** media (promedio), mediana (valor central), moda (valor más frecuente)
- **Dispersión:** desviación estándar, rango intercuartílico (IQR), mínimo y máximo
- Para categóricas: frecuencias, cardinalidad (número de categorías únicas)

⚠️ **Importante:** Debemos separar las filas de partido (`codigo_lista = 0`) de las de candidato (`codigo_lista ≥ 1`). Mezclarlas distorsionaría los promedios porque son unidades de análisis diferentes.

> **Pregunta guía:** ¿Cuántos votos obtiene un candidato típico en un municipio? ¿Y un partido? ¿La media y la mediana coinciden o difieren mucho?

In [ ]:
# ── SEPARAR FILAS DE PARTIDO Y CANDIDATO ──
df_partido = df[df['codigo_lista'] == 0].copy()
df_cand = df[df['codigo_lista'] > 0].copy()

print(f"Filas de PARTIDO (codigo_lista = 0):  {len(df_partido):>10,}")
print(f"Filas de CANDIDATO (codigo_lista ≥ 1): {len(df_cand):>10,}")
print(f"{'Total:':<42s} {len(df):>10,}")

In [ ]:
# ── DESCRIPTIVOS DE VOTOS: FILAS DE PARTIDO ──
print("═" * 80)
print("ESTADÍSTICA DESCRIPTIVA — VOTOS POR PARTIDO (en cada municipio)")
print("═" * 80)
print(df_partido['votos'].describe().to_string())
print(f"\n   Mediana:   {df_partido['votos'].median():,.0f}")
print(f"   Moda:      {df_partido['votos'].mode().values[0]:,}")
print(f"   Asimetría: {df_partido['votos'].skew():.2f}")

In [ ]:
# ── DESCRIPTIVOS DE VOTOS: FILAS DE CANDIDATO ──
print("═" * 80)
print("ESTADÍSTICA DESCRIPTIVA — VOTOS POR CANDIDATO (en cada municipio)")
print("═" * 80)
print(df_cand['votos'].describe().to_string())
print(f"\n   Mediana:   {df_cand['votos'].median():,.0f}")
print(f"   Moda:      {df_cand['votos'].mode().values[0]:,}")
print(f"   Asimetría: {df_cand['votos'].skew():.2f}")

In [ ]:
# ── FRECUENCIAS DE VARIABLES CATEGÓRICAS ──
print("═" * 80)
print("VARIABLES CATEGÓRICAS — FRECUENCIAS")
print("═" * 80)

print("\n📌 Circunscripción:")
print(df['circunscripcion'].value_counts().to_string())

print("\n📌 Top 10 departamentos (por número de registros):")
print(df['departamento'].value_counts().head(10).to_string())

# Número de partidos únicos (usando los nombres en las filas de partido)
n_partidos_nac = df_partido[df_partido['circunscripcion'] == 'Nacional']['nombres'].nunique()
n_partidos_ind = df_partido[df_partido['circunscripcion'] == 'Indígena']['nombres'].nunique()
print(f"\n📌 Partidos en circunscripción Nacional: {n_partidos_nac}")
print(f"📌 Partidos en circunscripción Indígena: {n_partidos_ind}")
print(f"📌 Municipios únicos: {df['municipio'].nunique()}")

### 🔍 Interpretación — Paso 5

Los valores descriptivos revelan un patrón fundamental:

- **La media de votos por candidato superá ampliamente a la mediana.** Esto indica una distribución **fuertemente sesgada a la derecha**: la gran mayoría obtiene pocos votos mientras un puñado de figuras concentra cifras enormes.
- La **asimetría (skewness)** es muy alta (>>0), confirmando el sesgo. Una distribución simétrica tendría skewness ≈ 0.
- La **moda es 0**: el valor más frecuente es cero votos, lo cual tiene sentido político — candidatos de listas pequeñas aparecen en todos los municipios pero solo reciben votos en algunos.

> **Reflexión política:** Esta distribución refleja la estructura del sistema electoral colombiano. Con decenas de partidos y cientos de candidatos compitiendo por 108 curules, la competencia sigue un patrón de «pocos ganadores y muchos perdedores». Veremos esto con más claridad en las visualizaciones.

## Paso 6 · Visualización univariada y detección de outliers
🟢 Básico

Las tablas numéricas del paso anterior dieron una idea general, pero una buena **visualización** revela patrones que los números solos no muestran. Crearemos:

- **Histogramas** para ver la forma de la distribución de votos
- **Boxplots** para visualizar la dispersión y detectar **outliers** (valores atípicos)
- **Gráficos de barras** para las variables categóricas principales

Un **outlier** es una observación inusualmente alejada del resto. No siempre es un error — a veces es la información más importante.

> **Pregunta guía:** ¿La distribución de votos es simétrica o hay unos pocos candidatos que concentran la mayoría? ¿Quiénes son los outliers y por qué lo son?

In [ ]:
# ── HISTOGRAMA DE VOTOS POR CANDIDATO ──
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Histograma completo
axes[0].hist(df_cand['votos'], bins=100, color=COLORES['principal'],
             edgecolor='white', alpha=0.85)
axes[0].set_title('Distribución de votos por candidato\n(por municipio)',
                  fontweight='bold')
axes[0].set_xlabel('Votos')
axes[0].set_ylabel('Frecuencia')
axes[0].axvline(df_cand['votos'].mean(), color=COLORES['acento'], linestyle='--',
                linewidth=2, label=f"Media: {df_cand['votos'].mean():,.0f}")
axes[0].axvline(df_cand['votos'].median(), color=COLORES['positivo'], linestyle='-',
                linewidth=2, label=f"Mediana: {df_cand['votos'].median():,.0f}")
axes[0].legend()

# Zoom: percentil 95
p95 = df_cand['votos'].quantile(0.95)
votos_zoom = df_cand[df_cand['votos'] <= p95]['votos']
axes[1].hist(votos_zoom, bins=80, color=COLORES['secundario'],
             edgecolor='white', alpha=0.85)
axes[1].set_title(f'Zoom: votos ≤ percentil 95 ({p95:,.0f})', fontweight='bold')
axes[1].set_xlabel('Votos')
axes[1].set_ylabel('Frecuencia')

plt.tight_layout()
plt.show()
print("💡 El histograma izquierdo está comprimido: casi toda la frecuencia se\n"
      "   concentra cerca de 0. El zoom derecho muestra la distribución real\n"
      "   de la gran mayoría de candidatos.")

In [ ]:
# ── BOXPLOT DE VOTOS POR CANDIDATO ──
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Boxplot general
axes[0].boxplot(df_cand['votos'].dropna(), vert=True, patch_artist=True,
                boxprops=dict(facecolor=COLORES['claro'], edgecolor=COLORES['principal']),
                medianprops=dict(color=COLORES['acento'], linewidth=2),
                flierprops=dict(marker='.', markerfacecolor=COLORES['negativo'],
                                markersize=2, alpha=0.3))
axes[0].set_title('Boxplot de votos por candidato\n(por municipio)', fontweight='bold')
axes[0].set_ylabel('Votos')

# Boxplot por circunscripción
data_circ = [df_cand[df_cand['circunscripcion'] == c]['votos'].dropna()
             for c in ['Nacional', 'Indígena']]
axes[1].boxplot(data_circ, labels=['Nacional', 'Indígena'], vert=True, patch_artist=True,
                boxprops=dict(facecolor=COLORES['claro'], edgecolor=COLORES['principal']),
                medianprops=dict(color=COLORES['acento'], linewidth=2),
                flierprops=dict(marker='.', markerfacecolor=COLORES['negativo'],
                                markersize=2, alpha=0.3))
axes[1].set_title('Boxplot por circunscripción', fontweight='bold')
axes[1].set_ylabel('Votos')

plt.tight_layout()
plt.show()

In [ ]:
# ── IDENTIFICAR Y NOMBRAR LOS OUTLIERS MÁS EXTREMOS ──
# Votos totales nacionales por candidato
votos_totales_cand = (
    df_cand
    .groupby(['primer_apellido', 'segundo_apellido', 'nombres'])
    .agg(votos_total=('votos', 'sum'), curules=('curules', 'max'))
    .reset_index()
    .sort_values('votos_total', ascending=False)
)
votos_totales_cand['candidato'] = (
    votos_totales_cand['nombres'] + ' ' +
    votos_totales_cand['primer_apellido'] + ' ' +
    votos_totales_cand['segundo_apellido']
)

print("═" * 80)
print("TOP 20 CANDIDATOS MÁS VOTADOS (total nacional)")
print("═" * 80)
top20 = votos_totales_cand.head(20)
for _, row in top20.iterrows():
    curul = "🏛️" if row['curules'] == 1 else "  "
    print(f"  {curul} {row['candidato']:<45s} {row['votos_total']:>10,} votos")

In [ ]:
# ── GRÁFICO DE BARRAS: TOP 15 CANDIDATOS ──
top15 = votos_totales_cand.head(15)

fig, ax = plt.subplots(figsize=(12, 8))
colores_barras = [COLORES['acento'] if c == 1 else COLORES['secundario']
                  for c in top15['curules']]
ax.barh(range(len(top15)), top15['votos_total'], color=colores_barras, edgecolor='white')
ax.set_yticks(range(len(top15)))
ax.set_yticklabels(top15['candidato'], fontsize=11)
ax.invert_yaxis()
ax.set_xlabel('Total de votos a nivel nacional')
ax.set_title('Top 15 candidatos más votados al Senado 2018',
             fontsize=14, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.annotate('🟠 Obtuvo curul    ⚪ No obtuvo curul',
            xy=(0.98, 0.02), xycoords='axes fraction', ha='right', fontsize=10,
            bbox=dict(boxstyle='round', facecolor=COLORES['claro']))
plt.tight_layout()
plt.show()

### 🔍 Interpretación — Paso 6

La distribución de votos por candidato es **extremadamente sesgada**. La gran mayoría de registros tiene pocos votos (muchos son cero), mientras unos pocos candidatos acumulan cifras enormes.

Los **outliers** más extremos no son errores de registro — son los **senadores más votados del país**. Eliminarlos sería eliminar la información más importante del dataset.

> **Decisión clave:** No eliminamos estos outliers. En datos electorales, los valores extremos *son* la historia: representan a los políticos con mayor capital electoral. Un análisis que los excluya estaría literalmente borrando a los actores más relevantes.

El contraste entre la circunscripción Nacional y la Indígena en volumen de votos ya es visible. Lo exploraremos con mayor profundidad adelante.

---
# 🟡 Nivel Intermedio — Limpieza, Interacciones y Contexto

**Objetivo:** ¿Cómo estandarizo la información y cómo se relacionan las variables entre sí para revelar verdaderos patrones?

En este nivel dejamos de solo *describir* los datos y empezamos a *interrogarlos*: buscamos relaciones entre variables, limpiamos inconsistencias y contextualizamos los hallazgos.

## Paso 7 · Tratamiento de valores faltantes e inconsistencias
🟡 Intermedio

Ahora que conocemos la estructura del dataset, debemos **estandarizarlo**. En datos de texto libre (nombres de municipios, partidos), es frecuente encontrar inconsistencias tipográficas: tildes faltantes, mayúsculas irregulares, espacios extra.

Vamos a:
1. **Documentar las decisiones** sobre nulos (ya resueltas en el Paso 3).
2. **Estandarizar nombres** de municipios y partidos para evitar que «MEDELLIN» y «Medellín» cuenten como dos entidades distintas.
3. **Crear una columna de nombre de partido** reutilizable para todo el análisis.

> **Pregunta guía:** ¿Hay inconsistencias en los nombres que podrían causar errores al agrupar datos?

In [29]:
# ── ESTANDARIZACIÓN DE NOMBRES ──
# El dataset original ya tiene todo en MAYÚSCULAS sin tildes.
# Documentamos esta decisión: mantenemos el formato original por consistencia.

# Verificar formato de municipios (¿todo mayúsculas?)
muestra_mun = df['municipio'].unique()[:10]
print("Muestra de municipios:", list(muestra_mun))

# Verificar formato de departamentos
muestra_dpto = df['departamento'].unique()[:10]
print("Muestra de departamentos:", list(muestra_dpto))

# Verificar espacios extra en nombres de partido
partidos_unicos = df_partido['nombres'].unique()
tiene_espacios = [p for p in partidos_unicos if '  ' in str(p)]
print(f"\nPartidos con espacios extra: {len(tiene_espacios)}")
if tiene_espacios:
    for p in tiene_espacios[:5]:
        print(f"  → '{p}'")

Muestra de municipios: ['MEDELLIN', 'ABEJORRAL', 'ABRIAQUI', 'ALEJANDRIA', 'AMAGA', 'AMALFI', 'ANDES', 'ANGELOPOLIS', 'ANGOSTURA', 'ANORI']
Muestra de departamentos: ['ANTIOQUIA', 'ATLANTICO', 'CONSULADOS', 'BOGOTA DC', 'BOLIVAR', 'BOYACA', 'CALDAS', 'CAQUETA', 'CAUCA', 'CESAR']

Partidos con espacios extra: 3
  → 'MOVIMIENTO AUTORIDADES INDIGENAS DE COLOMBIA  AICO'
  → 'PARTIDO SOCIAL DE UNIDAD NACIONAL  PARTIDO DE LA U'
  → 'MOVIMIENTO ALTERNATIVO INDIGENA Y SOCIAL  MAIS'


In [ ]:
# ── CREAR COLUMNA DE NOMBRE DE PARTIDO ──
# Mapeamos codigo_partido → nombre del partido usando las filas de partido
mapa_partidos = (
    df_partido
    .groupby('codigo_partido')['nombres']
    .first()
    .to_dict()
)

# Limpiar nombres de partido: quitar espacios dobles, estandarizar
mapa_partidos_limpio = {k: ' '.join(v.split()) for k, v in mapa_partidos.items()}
df['nombre_partido'] = df['codigo_partido'].map(mapa_partidos_limpio)

# Verificar la asignación
print("═" * 80)
print("PARTIDOS POLÍTICOS EN EL DATASET")
print("═" * 80)
for cod, nombre in sorted(mapa_partidos_limpio.items(), key=lambda x: x[1]):
    n_registros = (df['codigo_partido'] == cod).sum()
    print(f"  {cod:>12s} → {nombre:<55s} ({n_registros:,} registros)")

# Crear nombre completo del candidato (solo filas de candidato)
df['nombre_candidato'] = np.where(
    df['codigo_lista'] > 0,
    df['nombres'].fillna('') + ' ' + df['primer_apellido'].fillna('') + ' ' + df['segundo_apellido'].fillna(''),
    np.nan
)
df['nombre_candidato'] = df['nombre_candidato'].str.strip()

# Actualizar subconjuntos
df_partido = df[df['codigo_lista'] == 0].copy()
df_cand = df[df['codigo_lista'] > 0].copy()

print(f"\n✅ Columna 'nombre_partido' creada para todas las filas.")
print(f"✅ Columna 'nombre_candidato' creada para filas de candidato.")

### 🔍 Interpretación — Paso 7

El dataset original ya viene en **mayúsculas sin tildes**, que es el formato estándar de la Registraduría. Mantuvimos este formato por consistencia. Encontramos algunos partidos con espacios dobles internos (como `"MOVIMIENTO AUTORIDADES INDIGENAS DE COLOMBIA  AICO"`) que corregimos al crear el mapa de partidos.

Decisiones clave:
- **Nulos en `primer_apellido`/`segundo_apellido`:** No se imputan (son estructurales, MNAR).
- **Formato de texto:** Se mantiene en mayúsculas sin tildes, coherente con la fuente original.
- **Nuevas columnas:** `nombre_partido` (para todas las filas) y `nombre_candidato` (solo candidatos), que facilitan el análisis posterior.

> Los nombres de partidos como `"COALICION LISTA DE LA DECENCIA (ASI;UP;MAIS)"` reflejan las coaliciones electorales colombianas: varios movimientos que se presentan juntos bajo una sola lista.

## Paso 8 · Tratamiento de outliers
🟡 Intermedio

En el Paso 6 identificamos outliers en la distribución de votos. Ahora debemos decidir **qué hacer con ellos**, documentando nuestra justificación.

El método estadístico más común para detectar outliers es la **regla del IQR** (Rango Intercuartílico):
- Se calcula: `IQR = Q3 - Q1`
- Todo valor por debajo de `Q1 - 1.5 × IQR` o por encima de `Q3 + 1.5 × IQR` se marca como outlier.

Pero **cuidado**: ser un outlier estadístico no significa ser un dato erróneo.

> **Pregunta guía:** ¿Cuántos registros caen como "outlier estadístico" según la regla IQR? ¿Se trata de errores o de información legítima y valiosa?

In [ ]:
# ── ANÁLISIS DE OUTLIERS CON LA REGLA IQR ──
# Trabajamos solo con candidatos individuales y votos > 0 (excluimos los ceros)
votos_cand_positivos = df_cand[df_cand['votos'] > 0]['votos']

Q1 = votos_cand_positivos.quantile(0.25)
Q3 = votos_cand_positivos.quantile(0.75)
IQR = Q3 - Q1
limite_inferior = Q1 - 1.5 * IQR
limite_superior = Q3 + 1.5 * IQR

outliers_iqr = votos_cand_positivos[
    (votos_cand_positivos < limite_inferior) | (votos_cand_positivos > limite_superior)
]

print("═" * 80)
print("ANÁLISIS DE OUTLIERS — REGLA IQR")
print("═" * 80)
print(f"Q1 (percentil 25):     {Q1:>10,.0f}")
print(f"Q3 (percentil 75):     {Q3:>10,.0f}")
print(f"IQR (Q3 - Q1):         {IQR:>10,.0f}")
print(f"Límite inferior:       {limite_inferior:>10,.0f}")
print(f"Límite superior:       {limite_superior:>10,.0f}")
print(f"\nRegistros 'outlier' (votos > 0): {len(outliers_iqr):,} de {len(votos_cand_positivos):,}")
print(f"Porcentaje:            {len(outliers_iqr)/len(votos_cand_positivos)*100:.1f}%")

# Visualización
fig, ax = plt.subplots(figsize=(14, 4))
ax.hist(votos_cand_positivos, bins=150, color=COLORES['principal'],
        edgecolor='white', alpha=0.8, label='Dentro del rango')
ax.axvline(limite_superior, color=COLORES['negativo'], linestyle='--', linewidth=2,
           label=f'Límite IQR superior: {limite_superior:,.0f}')
ax.set_title('Distribución de votos por candidato (votos > 0) con límite IQR',
             fontweight='bold')
ax.set_xlabel('Votos')
ax.set_ylabel('Frecuencia')
ax.legend()
plt.tight_layout()
plt.show()

### 🔍 Interpretación — Paso 8

La regla del IQR clasifica una proporción notable de registros como "outliers". Pero examinando estos casos, vemos que incluyen a **los candidatos más votados del país**: senadores que obtuvieron su curul precisamente por esos votos.

> **Decisión: MANTENER todos los outliers.**
>
> **Justificación:**
> 1. No son errores de registro — son datos verificados por la Registraduría.
> 2. Son legítimos y sustantivamente importantes — representan a los senadores electos.
> 3. Eliminarlos sesgaría el análisis al borrar la información más relevante.
> 4. La distribución sesgada es una propiedad *real* de la competencia electoral, no un artefacto de los datos.

Esta es una lección importante: **las herramientas estadísticas no tienen contexto**. La regla del IQR es útil para detectar errores de captura en datos industriales, pero en datos electorales un "outlier" puede ser el presidente de la República. El analista debe aportar el juicio que la fórmula no puede.

## Paso 9 · Análisis bivariado: numérica vs. numérica
🟡 Intermedio

Hasta ahora hemos examinado una variable a la vez (análisis **univariado**). Ahora buscamos **relaciones entre pares de variables numéricas**. La herramienta principal es la **correlación**: un número entre -1 y +1 que mide la fuerza de la relación lineal.

⚠️ **Correlación ≠ Causalidad.** Que dos variables se muevan juntas no significa que una cause la otra.

Para tener variables numéricas comparables, agregaremos los datos a nivel **partido × departamento**, calculando: total de votos del partido y número de candidatos presentados.

> **Pregunta guía:** ¿Los partidos que presentan más candidatos obtienen más votos? ¿O la cantidad de candidatos no predice el éxito electoral?

In [ ]:
# ── AGREGAR DATOS A NIVEL PARTIDO-DEPARTAMENTO ──
# Para el análisis bivariado, necesitamos al menos dos variables numéricas
# Creamos un DataFrame agrupado a nivel partido-departamento

# Total de votos del partido (de las filas de partido, que son votos de lista)
votos_partido_dpto = (
    df_partido
    .groupby(['departamento', 'codigo_partido', 'nombre_partido'])
    .agg(votos_partido=('votos', 'sum'))
    .reset_index()
)

# Número de candidatos que presentó el partido en cada departamento
n_candidatos_dpto = (
    df_cand
    .groupby(['departamento', 'codigo_partido'])
    .agg(n_candidatos=('codigo_lista', 'nunique'))
    .reset_index()
)

# Votos totales de los candidatos individuales del partido
votos_cand_dpto = (
    df_cand
    .groupby(['departamento', 'codigo_partido'])
    .agg(votos_candidatos=('votos', 'sum'))
    .reset_index()
)

# Unir todo
biv = votos_partido_dpto.merge(n_candidatos_dpto, on=['departamento', 'codigo_partido'], how='left')
biv = biv.merge(votos_cand_dpto, on=['departamento', 'codigo_partido'], how='left')
biv['votos_total'] = biv['votos_partido'] + biv['votos_candidatos'].fillna(0)
biv['votos_por_candidato'] = biv['votos_total'] / biv['n_candidatos'].replace(0, np.nan)

print(f"Tabla bivariada: {biv.shape[0]} filas (partido × departamento)")
biv.head(10)

In [ ]:
# ── CORRELACIÓN Y SCATTER PLOT ──
# Correlación entre votos totales y número de candidatos
corr_pearson = biv[['votos_total', 'n_candidatos']].corr().iloc[0, 1]
corr_spearman = biv[['votos_total', 'n_candidatos']].corr(method='spearman').iloc[0, 1]

print(f"Correlación Pearson  (votos_total vs n_candidatos): {corr_pearson:.3f}")
print(f"Correlación Spearman (votos_total vs n_candidatos): {corr_spearman:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter: votos_total vs n_candidatos
axes[0].scatter(biv['n_candidatos'], biv['votos_total'],
                alpha=0.4, color=COLORES['principal'], s=20)
axes[0].set_xlabel('Número de candidatos presentados')
axes[0].set_ylabel('Votos totales del partido en el departamento')
axes[0].set_title(f'Votos vs. Candidatos presentados\n(Pearson r = {corr_pearson:.2f})',
                  fontweight='bold')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# Heatmap de correlaciones
cols_num = ['votos_partido', 'votos_candidatos', 'votos_total', 'n_candidatos', 'votos_por_candidato']
corr_matrix = biv[cols_num].corr(method='spearman')
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=axes[1], square=True,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
axes[1].set_title('Matriz de correlación (Spearman)', fontweight='bold')

plt.tight_layout()
plt.show()

### 🔍 Interpretación — Paso 9

La correlación entre el número de candidatos y los votos totales nos dice si «presentar más candidatos» se traduce en más votos. La correlación positiva indica que los partidos más grandes tienden tanto a presentar más candidatos como a obtener más votos — pero esto no significa que presentar muchos candidatos *cause* más votos. Más bien refleja que los partidos con mayor estructura organizativa tienen **ambas cosas a la vez**.

> **Reflexión:** En ciencia política, esto se relaciona con el concepto de **estructura partidista**. Los partidos con mayor implantación territorial presentan más candidatos *y* movilizan más electores. La variable latente es la fortaleza organizativa del partido.

## Paso 10 · Análisis bivariado: categórica y mixta
🟡 Intermedio

Ahora cruzamos variables **categóricas** entre sí (tablas de contingencia) y categóricas con numéricas (boxplots agrupados). Esto nos permite responder preguntas como: ¿la distribución de votos varía entre departamentos? ¿Qué partidos dominan en cada circunscripción?

> **Pregunta guía:** ¿Cómo varía la distribución de votos entre los departamentos más grandes del país? ¿Qué partidos dominan la circunscripción Nacional vs. la Indígena?

In [ ]:
# ── BOXPLOTS DE VOTOS POR DEPARTAMENTO (Top 10) ──
# Seleccionar los 10 departamentos con más votos totales
top10_dptos = (
    df_cand.groupby('departamento')['votos']
    .sum()
    .nlargest(10)
    .index
    .tolist()
)

df_top10 = df_cand[df_cand['departamento'].isin(top10_dptos)]

fig, ax = plt.subplots(figsize=(16, 7))
orden = (df_top10.groupby('departamento')['votos'].median()
         .sort_values(ascending=False).index)
sns.boxplot(data=df_top10, x='departamento', y='votos', order=orden,
            palette='Blues_d', showfliers=False, ax=ax)
ax.set_title('Distribución de votos por candidato — Top 10 departamentos\n(sin outliers para legibilidad)',
             fontweight='bold')
ax.set_xlabel('Departamento')
ax.set_ylabel('Votos por candidato (por municipio)')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# ── TABLA DE CONTINGENCIA: CIRCUNSCRIPCIÓN × PARTIDOS PRINCIPALES ──
# Top 8 partidos por votos totales
top_partidos = (
    df_partido.groupby('nombre_partido')['votos']
    .sum()
    .nlargest(8)
    .index
    .tolist()
)

# Tabla cruzada
contingencia = pd.crosstab(
    df_partido[df_partido['nombre_partido'].isin(top_partidos)]['nombre_partido'],
    df_partido[df_partido['nombre_partido'].isin(top_partidos)]['circunscripcion'],
    values=df_partido[df_partido['nombre_partido'].isin(top_partidos)]['votos'],
    aggfunc='sum',
    margins=True
)

print("═" * 80)
print("TABLA DE CONTINGENCIA: VOTOS POR CIRCUNSCRIPCIÓN")
print("═" * 80)
# Formatear con separadores de miles
print(contingencia.map(lambda x: f'{x:,.0f}' if pd.notnull(x) else ''))

In [ ]:
# ── HEATMAP DE LA TABLA DE CONTINGENCIA ──
# Sin la fila/columna 'All'
cont_sin_margins = contingencia.drop('All', axis=0).drop('All', axis=1)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cont_sin_margins, annot=True, fmt=',.0f', cmap='YlOrRd',
            linewidths=0.5, ax=ax, cbar_kws={'label': 'Votos totales'})
ax.set_title('Votos por partido × circunscripción', fontweight='bold')
ax.set_ylabel('Partido')
ax.set_xlabel('Circunscripción')
plt.tight_layout()
plt.show()

### 🔍 Interpretación — Paso 10

El análisis bivariado revela diferencias territoriales significativas:

- Los departamentos con mayor población (Bogotá, Antioquia, Valle del Cauca) tienen, como es esperable, más votos por candidato. Pero la **dispersión** interna es enorme: incluso en Antioquia, la mayoría de candidatos reciben pocos votos.
- La tabla de contingencia muestra que los grandes partidos concentran casi toda su votación en la circunscripción **Nacional**, mientras que la circunscripción **Indígena** tiene partidos completamente diferentes (movimientos étnicos y comunitarios).

> **Reflexión política:** Las dos circunscripciones son esencialmente **elecciones paralelas** con actores distintos. Analizarlas como un solo bloque enmascararía estas diferencias fundamentales.

## Paso 11 · Análisis de distribuciones
🟡 Intermedio

En el Paso 6 vimos que la distribución de votos es extremadamente sesgada. Ahora vamos a **diagnosticar formalmente** esa distribución y a aplicar una **transformación logarítmica** para hacerla más simétrica.

**¿Por qué importa la forma de la distribución?**
- Muchos tests estadísticos y modelos de ML asumen o se benefician de distribuciones simétricas (cercanas a la «normal» o campana de Gauss).
- La transformación logarítmica es particularmente adecuada para datos que siguen una **distribución log-normal**: fenómenos donde hay muchos valores pequeños y pocos valores muy grandes. Esto es típico en competencia electoral, ingresos económicos, tamaños de ciudades, etc.

Un **QQ-plot** (*Quantile-Quantile plot*) compara los cuantiles de nuestros datos contra los cuantiles teóricos de una distribución normal. Si los puntos caen sobre la línea diagonal, la distribución es normal.

> **Pregunta guía:** ¿La distribución de votos es normal? Y después de la transformación logarítmica, ¿se acerca más?

In [ ]:
# ── DIAGNÓSTICO DE DISTRIBUCIÓN ──
votos_positivos = df_cand[df_cand['votos'] > 0]['votos']
votos_log = np.log1p(votos_positivos)  # log(1 + x) para manejar valores pequeños

print("═" * 80)
print("MÉTRICAS DE FORMA DE LA DISTRIBUCIÓN")
print("═" * 80)
print(f"{'Métrica':<25s} {'Original':>15s} {'Log-transformada':>15s}")
print("─" * 55)
print(f"{'Asimetría (skewness)':<25s} {votos_positivos.skew():>15.2f} {votos_log.skew():>15.2f}")
print(f"{'Curtosis':<25s} {votos_positivos.kurtosis():>15.2f} {votos_log.kurtosis():>15.2f}")
print(f"{'Media':<25s} {votos_positivos.mean():>15,.0f} {votos_log.mean():>15.2f}")
print(f"{'Mediana':<25s} {votos_positivos.median():>15,.0f} {votos_log.median():>15.2f}")

# Histogramas comparativos
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].hist(votos_positivos, bins=100, color=COLORES['principal'],
             edgecolor='white', alpha=0.85, density=True)
axes[0].set_title('Distribución original de votos\n(candidatos con votos > 0)',
                  fontweight='bold')
axes[0].set_xlabel('Votos')
axes[0].set_ylabel('Densidad')

axes[1].hist(votos_log, bins=80, color=COLORES['positivo'],
             edgecolor='white', alpha=0.85, density=True)
# Superponer curva normal teórica
x_normal = np.linspace(votos_log.min(), votos_log.max(), 100)
axes[1].plot(x_normal, stats.norm.pdf(x_normal, votos_log.mean(), votos_log.std()),
             color=COLORES['negativo'], linewidth=2, linestyle='--', label='Normal teórica')
axes[1].set_title('Distribución log-transformada\nlog(1 + votos)', fontweight='bold')
axes[1].set_xlabel('log(1 + votos)')
axes[1].set_ylabel('Densidad')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── QQ-PLOT ──
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# QQ-plot de los datos originales
stats.probplot(votos_positivos.sample(5000, random_state=42), dist="norm", plot=axes[0])
axes[0].set_title('QQ-Plot: votos originales vs. Normal', fontweight='bold')
axes[0].get_lines()[0].set(color=COLORES['principal'], markersize=3, alpha=0.5)
axes[0].get_lines()[1].set(color=COLORES['negativo'], linewidth=2)

# QQ-plot de los datos log-transformados
stats.probplot(votos_log.sample(5000, random_state=42), dist="norm", plot=axes[1])
axes[1].set_title('QQ-Plot: log(votos) vs. Normal', fontweight='bold')
axes[1].get_lines()[0].set(color=COLORES['positivo'], markersize=3, alpha=0.5)
axes[1].get_lines()[1].set(color=COLORES['negativo'], linewidth=2)

plt.tight_layout()
plt.show()

### 🔍 Interpretación — Paso 11

La transformación logarítmica **reduce drásticamente la asimetría** y produce una distribución mucho más cercana a la normal:
- La asimetría pasa de un valor muy alto a uno cercano a cero.
- El QQ-plot de los datos log-transformados se alinea mucho mejor con la recta de referencia (distribución normal teórica).

Esto confirma que los votos siguen aproximadamente una **distribución log-normal**, que es típica en fenómenos de competencia donde hay:
- **Rendimientos crecientes**: los candidatos conocidos atraen más atención, lo cual atrae más votos (efecto «bola de nieve»).
- **Restricciones de no-negatividad**: los votos no pueden ser negativos.

> **Reflexión:** La distribución log-normal aparece en muchos fenómenos sociales: ingresos, tamaño de ciudades, citas académicas, número de seguidores en redes sociales. Todos comparten una estructura de «pocos concentran mucho». En elecciones, esto refleja la **ley de hierro de la oligarquía** de Michels: la competencia tiende naturalmente a la concentración.

## Paso 12 · Segmentación por subgrupos
🟡 Intermedio

Este es uno de los pasos más importantes del EDA: **¿los patrones generales se sostienen cuando desagregamos por subgrupos?**

La **paradoja de Simpson** es un fenómeno estadístico donde una tendencia que existe en los datos agregados **desaparece o se invierte** cuando se analiza dentro de subgrupos. Es como si las estadísticas nacionales de empleo mostraran mejora, pero al mirar región por región todas estuvieran empeorando.

Vamos a segmentar por la variable más relevante: **circunscripción Nacional vs. Indígena**.

> **Pregunta guía:** ¿Las conclusiones que sacamos del dataset completo se sostienen cuando separamos las dos circunscripciones? ¿Hay algún patrón que cambie o se invierta?

In [ ]:
# ── COMPARACIÓN NACIONAL VS. INDÍGENA ──
print("═" * 80)
print("SEGMENTACIÓN: CIRCUNSCRIPCIÓN NACIONAL vs. INDÍGENA")
print("═" * 80)

for circ in ['Nacional', 'Indígena']:
    sub = df_cand[df_cand['circunscripcion'] == circ]
    sub_p = df_partido[df_partido['circunscripcion'] == circ]
    print(f"\n{'─' * 40}")
    print(f"📌 Circunscripción: {circ.upper()}")
    print(f"{'─' * 40}")
    print(f"  Registros de candidato:  {len(sub):>10,}")
    print(f"  Candidatos únicos:       {sub['nombre_candidato'].nunique():>10,}")
    print(f"  Partidos únicos:         {sub_p['nombre_partido'].nunique():>10,}")
    print(f"  Municipios:              {sub['municipio'].nunique():>10,}")
    print(f"  Media de votos/cand/mun: {sub['votos'].mean():>10,.1f}")
    print(f"  Mediana:                 {sub['votos'].median():>10,.1f}")
    print(f"  Máximo:                  {sub['votos'].max():>10,}")
    # % de ceros
    pct_ceros = (sub['votos'] == 0).mean() * 100
    print(f"  % registros con 0 votos: {pct_ceros:>9.1f}%")

In [ ]:
# ── VISUALIZACIÓN COMPARATIVA ──
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# 1. Histogramas superpuestos (log-transformados)
for circ, color, label in [('Nacional', COLORES['principal'], 'Nacional'),
                            ('Indígena', COLORES['acento'], 'Indígena')]:
    votos_circ = df_cand[(df_cand['circunscripcion'] == circ) & (df_cand['votos'] > 0)]['votos']
    axes[0].hist(np.log1p(votos_circ), bins=60, alpha=0.6, color=color,
                 label=label, density=True, edgecolor='white')
axes[0].set_title('Distribución log(votos)\npor circunscripción', fontweight='bold')
axes[0].set_xlabel('log(1 + votos)')
axes[0].set_ylabel('Densidad')
axes[0].legend()

# 2. Votos totales por partido (top 5 de cada circunscripción)
for idx, circ in enumerate(['Nacional', 'Indígena']):
    sub = df_partido[df_partido['circunscripcion'] == circ]
    top5 = sub.groupby('nombre_partido')['votos'].sum().nlargest(5)
    ax = axes[idx + 1]
    bars = ax.barh(range(len(top5)), top5.values,
                   color=COLORES['principal'] if circ == 'Nacional' else COLORES['acento'],
                   edgecolor='white')
    ax.set_yticks(range(len(top5)))
    ax.set_yticklabels([n[:35] for n in top5.index], fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel('Votos totales')
    ax.set_title(f'Top 5 partidos\n{circ}', fontweight='bold')
    ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

plt.tight_layout()
plt.show()

### 🔍 Interpretación — Paso 12

La segmentación revela que las dos circunscripciones son **mundos electorales completamente distintos**:

| Aspecto | Nacional | Indígena |
|---------|----------|----------|
| Volumen de votos | Cientos de miles por candidato top | Decenas de miles |
| Partidos dominantes | Centro Democrático, Cambio Radical, etc. | AICO, MAIS, Alianza Social Independiente |
| Concentración del voto | Alta (pocos candidatos muy votados) | Más dispersa |
| Proporción de ceros | Alta | Muy alta |

Si hubiéramos analizado el dataset sin separar circunscripciones, los patrones de la Nacional (que tiene muchas más filas) habrían **enmascarado completamente** la dinámica de la Indígena.

> **Reflexión política:** La circunscripción especial indígena fue creada por la Constitución de 1991 para garantizar representación de los pueblos indígenas en el Senado (2 curules de las 108). Tiene partidos, dinámicas y escalas de votación completamente diferentes. Analizarla con los mismos parámetros que la Nacional sería un error analítico y político.

## Paso 13 · Exploración temporal
🟡 Intermedio — ⚠️ NO APLICA A ESTE DATASET

Este paso analiza **tendencias, estacionalidad y quiebres en el tiempo**. Sin embargo, nuestro dataset contiene datos de **una sola elección** (Senado 2018), por lo que no hay dimensión temporal que explorar.

**¿Cuándo se activaría este paso?** Si tuviéramos datos de múltiples elecciones (por ejemplo, Senado 2006, 2010, 2014, 2018, 2022), podríamos analizar:
- ¿Cómo ha evolucionado la concentración del voto?
- ¿Hay partidos que crecen o declinan sistemáticamente?
- ¿El número efectivo de partidos aumenta o disminuye con las reformas electorales?
- ¿La participación electoral tiene tendencia o es cíclica?

> 💡 **Ejercicio sugerido para el estudiante:** Descarga datos de elecciones anteriores del portal de la Registraduría y construye un análisis temporal comparado.

## Paso 14 · Exploración geoespacial
🟡 Intermedio

Colombia tiene 32 departamentos y más de 1,100 municipios. Las dinámicas electorales suelen tener un fuerte componente territorial: los partidos tienen bastiones regionales, y la participación varía enormemente entre zonas urbanas y rurales.

Lo ideal sería crear un **mapa coroplético** (un mapa donde cada departamento/municipio se colorea según una variable). Esto requiere la librería `geopandas` y un archivo de geometrías (shapefile) de Colombia. Como alternativa práctica, crearemos un **gráfico de barras horizontal** ordenado que funcione como «mapa conceptual» del territorio.

> **Pregunta guía:** ¿Cómo se distribuyen los votos geográficamente? ¿Hay departamentos que concentran desproporcionadamente la votación?

In [ ]:
# ── VOTOS POR DEPARTAMENTO ──
votos_dpto = (
    df_partido
    .groupby('departamento')['votos']
    .sum()
    .sort_values(ascending=True)
)

# Gráfico de barras horizontal (funciona como "mapa conceptual" del territorio)
fig, ax = plt.subplots(figsize=(12, 14))

colores = [COLORES['acento'] if d in votos_dpto.nlargest(5).index else COLORES['principal']
           for d in votos_dpto.index]

ax.barh(range(len(votos_dpto)), votos_dpto.values, color=colores, edgecolor='white')
ax.set_yticks(range(len(votos_dpto)))
ax.set_yticklabels(votos_dpto.index, fontsize=9)
ax.set_xlabel('Votos totales (partidos)')
ax.set_title('Votos totales al Senado 2018 por departamento\n(🟠 = Top 5 departamentos)',
             fontsize=14, fontweight='bold')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))

# Agregar valores al final de las barras para los top 5
for i, (nombre, votos) in enumerate(votos_dpto.items()):
    if nombre in votos_dpto.nlargest(5).index:
        ax.text(votos + votos_dpto.max() * 0.01, i, f'{votos:,.0f}',
                va='center', fontsize=8, color=COLORES['acento'])

plt.tight_layout()
plt.show()

In [ ]:
# ── BONUS: MAPA COROPLÉTICO (opcional, requiere geopandas) ──
# pip install geopandas
try:
    import geopandas as gpd
    # Si tienes un shapefile de departamentos de Colombia:
    # gdf = gpd.read_file('ruta/a/departamentos_colombia.shp')
    # votos_mapa = votos_dpto.reset_index()
    # votos_mapa.columns = ['departamento', 'votos']
    # gdf = gdf.merge(votos_mapa, left_on='NOMBRE_DPT', right_on='departamento')
    # gdf.plot(column='votos', cmap='YlOrRd', legend=True, figsize=(10, 12))
    print("⚠️ geopandas está instalado. Para crear un mapa coroplético,")
    print("   necesitas un shapefile de departamentos de Colombia (DANE/IGAC).")
    print("   El código está preparado arriba (comentado) para cuando tengas el shapefile.")
except ImportError:
    print("ℹ️ geopandas no está instalado. El gráfico de barras de arriba")
    print("  funciona como alternativa visual al mapa coroplético.")
    print("  Para instalar: pip install geopandas")

### 🔍 Interpretación — Paso 14

La distribución geográfica del voto muestra una **fuerte concentración territorial**:
- Los 5 departamentos más poblados (Bogotá, Antioquia, Valle del Cauca, Atlántico, Santander) concentran una proporción muy significativa del total de votos al Senado.
- Departamentos como Vaupés, Guainía, Vichada y Amazonas tienen volúmenes de votación mínimos — coherente con sus poblaciones pequeñas.

> **Reflexión política:** Esta concentración geográfica del voto tiene implicaciones profundas para la representación: los candidatos al Senado necesitan ser competitivos en los grandes centros urbanos para ganar curul. Esto puede marginar las agendas de los departamentos periféricos, que terminan subrepresentados en las prioridades legislativas.

---
# 🔴 Nivel Avanzado — Estructura Latente y Preparación para el Modelo

**Objetivo:** ¿Qué patrones invisibles existen en los datos, hay sesgos en la recolección y cómo adecuamos los datos para el Machine Learning?

En este nivel usamos herramientas más sofisticadas: creación de nuevas variables, reducción de dimensionalidad, clustering, tests de hipótesis y evaluación de sesgos.

## Paso 15 · Ingeniería de características (Feature Engineering)
🔴 Avanzado

**Ingeniería de características** (o *Feature Engineering*) es el proceso de crear nuevas variables a partir de las existentes para capturar patrones más complejos. Es como construir indicadores compuestos: el PIB per cápita es más informativo que el PIB solo o la población sola.

Vamos a crear variables a nivel de **municipio** que capturen diferentes dimensiones de la competencia electoral:

| Variable | Significado |
|----------|-------------|
| `votos_pct` | % de votos de cada partido sobre el total del municipio |
| `n_partidos_municipio` | Número de partidos que compitieron |
| `hhi_municipio` | Índice Herfindahl-Hirschman: concentración del voto |
| `tiene_circ_indigena` | Si el municipio tuvo circunscripción indígena |
| `votos_por_candidato` | Ratio votos / candidatos por partido |

El **Índice Herfindahl-Hirschman (HHI)** mide la concentración de un mercado (o de un sistema electoral). Se calcula sumando los cuadrados de las cuotas de participación. Va de 0 (competencia perfecta) a 10,000 (monopolio absoluto).

> **Pregunta guía:** ¿Podemos construir indicadores municipales que resuman la dinámica electoral local?

In [ ]:
# ── FEATURE ENGINEERING A NIVEL MUNICIPAL ──
# Trabajamos con filas de partido para las métricas municipales (circunscripción Nacional)
df_nac_partido = df_partido[df_partido['circunscripcion'] == 'Nacional'].copy()

# 1. Total de votos por municipio
total_mun = df_nac_partido.groupby(['codmpio', 'municipio', 'departamento'])['votos'].sum().reset_index()
total_mun.columns = ['codmpio', 'municipio', 'departamento', 'votos_total_municipio']

# 2. Porcentaje de votos por partido en cada municipio
df_nac_partido = df_nac_partido.merge(total_mun[['codmpio', 'votos_total_municipio']], on='codmpio')
df_nac_partido['votos_pct'] = (
    df_nac_partido['votos'] / df_nac_partido['votos_total_municipio'].replace(0, np.nan) * 100
)

# 3. Número de partidos por municipio (partidos con al menos 1 voto)
n_partidos = (
    df_nac_partido[df_nac_partido['votos'] > 0]
    .groupby('codmpio')['codigo_partido']
    .nunique()
    .reset_index()
)
n_partidos.columns = ['codmpio', 'n_partidos_municipio']

# 4. HHI por municipio
hhi = (
    df_nac_partido
    .groupby('codmpio')
    .apply(lambda g: ((g['votos'] / g['votos'].sum() * 100) ** 2).sum()
           if g['votos'].sum() > 0 else np.nan)
    .reset_index()
)
hhi.columns = ['codmpio', 'hhi_municipio']

# 5. Indicador de circunscripción indígena
mun_indigenas = df[df['circunscripcion'] == 'Indígena']['codmpio'].unique()
total_mun['tiene_circ_indigena'] = total_mun['codmpio'].isin(mun_indigenas).astype(int)

# 6. Votos por candidato (nivel partido-municipio)
n_cand_por_mun = (
    df_cand[df_cand['circunscripcion'] == 'Nacional']
    .groupby('codmpio')['codigo_lista']
    .nunique()
    .reset_index()
)
n_cand_por_mun.columns = ['codmpio', 'n_candidatos_municipio']

# ── ENSAMBLAR TABLA DE CARACTERÍSTICAS MUNICIPALES ──
mun_features = total_mun.copy()
mun_features = mun_features.merge(n_partidos, on='codmpio', how='left')
mun_features = mun_features.merge(hhi, on='codmpio', how='left')
mun_features = mun_features.merge(n_cand_por_mun, on='codmpio', how='left')
mun_features['votos_por_candidato'] = (
    mun_features['votos_total_municipio'] / mun_features['n_candidatos_municipio'].replace(0, np.nan)
)

print(f"Tabla de características municipales: {mun_features.shape}")
print("\nPrimeras filas:")
mun_features.head(10)

In [ ]:
# ── ESTADÍSTICAS DE LAS VARIABLES DERIVADAS ──
cols_features = ['votos_total_municipio', 'n_partidos_municipio', 'hhi_municipio',
                 'n_candidatos_municipio', 'votos_por_candidato', 'tiene_circ_indigena']
print("═" * 80)
print("ESTADÍSTICAS DE VARIABLES DERIVADAS (nivel municipal)")
print("═" * 80)
print(mun_features[cols_features].describe().round(2).to_string())

# Distribución del HHI
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(mun_features['hhi_municipio'].dropna(), bins=50,
             color=COLORES['principal'], edgecolor='white')
axes[0].set_title('Distribución del HHI municipal', fontweight='bold')
axes[0].set_xlabel('HHI (0 = disperso, 10000 = concentrado)')
axes[0].set_ylabel('Frecuencia')

axes[1].hist(mun_features['n_partidos_municipio'].dropna(), bins=30,
             color=COLORES['acento'], edgecolor='white')
axes[1].set_title('Número de partidos por municipio', fontweight='bold')
axes[1].set_xlabel('N° de partidos')
axes[1].set_ylabel('Frecuencia')

axes[2].scatter(mun_features['n_partidos_municipio'],
                mun_features['hhi_municipio'],
                alpha=0.4, s=15, color=COLORES['principal'])
axes[2].set_title('HHI vs. N° de partidos', fontweight='bold')
axes[2].set_xlabel('N° de partidos')
axes[2].set_ylabel('HHI')

plt.tight_layout()
plt.show()

### 🔍 Interpretación — Paso 15

Las variables derivadas revelan patrones municipales interesantes:

- **HHI:** La mayoría de municipios tienen un HHI relativamente bajo (competencia dispersa), pero hay municipios con concentración muy alta donde uno o dos partidos dominan.
- **Relación inversa HHI vs. N° partidos:** Como era de esperar, a más partidos compitiendo, menor concentración del voto.
- **Municipios con circunscripción indígena:** Son una minoría, identificables como subgrupo diferenciado.

> Estas variables nos servirán como insumo para PCA y clustering en los próximos pasos.

## Paso 16 · Análisis multivariado y reducción de dimensionalidad (PCA)
🔴 Avanzado

Cuando tenemos muchas variables, es difícil visualizar sus relaciones simultáneamente. El **Análisis de Componentes Principales (PCA)** es una técnica que comprime múltiples variables en unas pocas «componentes» que capturan la mayor parte de la variación en los datos.

> **Analogía:** Imagina que describes un municipio con 6 indicadores (votos, partidos, concentración, etc.). PCA busca si existe un «eje principal» que resuma la mayor parte de esas diferencias. Quizás ese eje separa municipios grandes y competitivos de municipios pequeños y dominados por un solo partido.

> **Pregunta guía:** ¿Se pueden reducir nuestras variables municipales a 2 o 3 dimensiones sin perder mucha información? ¿Qué significan esas dimensiones en términos políticos?

In [ ]:
# ── PCA SOBRE VARIABLES MUNICIPALES ──
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Seleccionar variables numéricas y eliminar nulos
cols_pca = ['votos_total_municipio', 'n_partidos_municipio', 'hhi_municipio',
            'n_candidatos_municipio', 'votos_por_candidato']

mun_pca = mun_features[cols_pca + ['municipio', 'departamento', 'tiene_circ_indigena']].dropna()
X = mun_pca[cols_pca].values

# Estandarizar (media=0, std=1) — imprescindible para PCA
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Aplicar PCA
pca = PCA()
X_pca = pca.fit_transform(X_scaled)

# Varianza explicada
print("═" * 80)
print("VARIANZA EXPLICADA POR COMPONENTE")
print("═" * 80)
for i, (var, cum) in enumerate(zip(pca.explained_variance_ratio_,
                                    np.cumsum(pca.explained_variance_ratio_))):
    barra = "█" * int(var * 50)
    print(f"  PC{i+1}: {var:.1%} (acumulada: {cum:.1%}) {barra}")

# Contribución de cada variable a los componentes
loadings = pd.DataFrame(
    pca.components_.T,
    columns=[f'PC{i+1}' for i in range(len(cols_pca))],
    index=cols_pca
)
print("\n═" * 80)
print("CONTRIBUCIÓN DE CADA VARIABLE (LOADINGS)")
print("═" * 80)
print(loadings.round(3).to_string())

In [ ]:
# ── BIPLOT: PROYECCIÓN 2D CON VECTORES DE VARIABLES ──
fig, ax = plt.subplots(figsize=(12, 10))

# Puntos (municipios) coloreados por presencia indígena
colores_mun = [COLORES['acento'] if ci == 1 else COLORES['principal']
               for ci in mun_pca['tiene_circ_indigena']]
ax.scatter(X_pca[:, 0], X_pca[:, 1], c=colores_mun, alpha=0.4, s=15)

# Vectores de variables
for i, var in enumerate(cols_pca):
    ax.arrow(0, 0, pca.components_[0, i] * 3, pca.components_[1, i] * 3,
             head_width=0.08, head_length=0.05, fc=COLORES['negativo'], ec=COLORES['negativo'])
    ax.text(pca.components_[0, i] * 3.3, pca.components_[1, i] * 3.3,
            var.replace('_', '\n'), fontsize=9, fontweight='bold',
            ha='center', va='center', color=COLORES['negativo'])

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} varianza)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} varianza)')
ax.set_title('Biplot PCA: Municipios en 2 dimensiones\n(🟠 = municipio con circ. indígena)',
             fontsize=14, fontweight='bold')
ax.axhline(0, color='gray', linewidth=0.5, linestyle='--')
ax.axvline(0, color='gray', linewidth=0.5, linestyle='--')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 🔍 Interpretación — Paso 16

El PCA comprime las 5 variables municipales en componentes interpretables:

- **PC1 (primer componente):** Captura principalmente el **tamaño del municipio** — votos totales, número de candidatos y votos por candidato apuntan en la misma dirección. Los municipios grandes están a un extremo, los pequeños al otro.
- **PC2 (segundo componente):** Captura la **concentración vs. dispersión** del voto — el HHI y el número de partidos apuntan en direcciones opuestas.

En términos políticos: la principal fuente de variación entre municipios es su **tamaño** (que determina cuántos votos, candidatos y partidos hay), y la segunda es la **estructura de competencia** (concentrada vs. fragmentada).

> Los municipios con circunscripción indígena (en naranja) tienden a agruparse en una zona del espacio, lo que sugiere que comparten características electorales distintivas.

## Paso 17 · Clustering exploratorio
🔴 Avanzado

El **clustering** busca agrupar municipios similares sin etiquetas previas. Usamos **K-Means**, un algoritmo que divide los datos en K grupos minimizando la distancia de cada punto al centro de su grupo.

Para elegir el número de clusters (K), usamos dos métodos:
- **Método del codo:** Graficar la "inercia" (suma de distancias al centro) vs. K. Buscamos el punto donde añadir más clusters deja de mejorar significativamente.
- **Coeficiente de silueta:** Mide qué tan bien asignado está cada punto a su cluster. Va de -1 (mal asignado) a +1 (perfectamente asignado).

> **Pregunta guía:** ¿Hay agrupaciones naturales de municipios según sus características electorales? ¿Son municipios de voto concentrado vs. fragmentado? ¿Grandes vs. pequeños?

In [ ]:
# ── MÉTODO DEL CODO Y SILUETA ──
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Reutilizamos X_scaled del paso de PCA
K_range = range(2, 11)
inercias = []
siluetas = []

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inercias.append(km.inertia_)
    siluetas.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Método del codo
axes[0].plot(K_range, inercias, 'o-', color=COLORES['principal'], linewidth=2)
axes[0].set_title('Método del codo', fontweight='bold')
axes[0].set_xlabel('Número de clusters (K)')
axes[0].set_ylabel('Inercia')

# Silueta
axes[1].plot(K_range, siluetas, 's-', color=COLORES['acento'], linewidth=2)
axes[1].set_title('Coeficiente de silueta', fontweight='bold')
axes[1].set_xlabel('Número de clusters (K)')
axes[1].set_ylabel('Silueta promedio')

plt.tight_layout()
plt.show()

# Elegir K óptimo
k_optimo = list(K_range)[np.argmax(siluetas)]
print(f"\nK con mejor silueta: {k_optimo} (silueta = {max(siluetas):.3f})")

In [ ]:
# ── APLICAR K-MEANS CON K ÓPTIMO ──
km_final = KMeans(n_clusters=k_optimo, random_state=42, n_init=10)
mun_pca['cluster'] = km_final.fit_predict(X_scaled)

# Visualizar clusters en el espacio PCA
fig, ax = plt.subplots(figsize=(12, 8))
colores_cluster = sns.color_palette('Set2', k_optimo)
for cl in range(k_optimo):
    mask = mun_pca['cluster'] == cl
    ax.scatter(X_pca[mask, 0], X_pca[mask, 1],
               c=[colores_cluster[cl]], label=f'Cluster {cl}',
               alpha=0.5, s=20)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} varianza)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} varianza)')
ax.set_title(f'Clusters de municipios (K={k_optimo}) en espacio PCA',
             fontsize=14, fontweight='bold')
ax.legend(title='Cluster')
plt.tight_layout()
plt.show()

In [ ]:
# ── PERFILAMIENTO DE CLUSTERS ──
print("═" * 80)
print(f"PERFILAMIENTO DE CLUSTERS (K = {k_optimo})")
print("═" * 80)

perfil = mun_pca.groupby('cluster')[cols_pca + ['tiene_circ_indigena']].agg(['mean', 'median', 'count'])
for cl in range(k_optimo):
    sub = mun_pca[mun_pca['cluster'] == cl]
    print(f"\n{'═' * 60}")
    print(f"CLUSTER {cl} — {len(sub)} municipios ({len(sub)/len(mun_pca)*100:.1f}%)")
    print(f"{'═' * 60}")
    print(f"  Votos totales (media):         {sub['votos_total_municipio'].mean():>12,.0f}")
    print(f"  N° partidos (media):           {sub['n_partidos_municipio'].mean():>12.1f}")
    print(f"  HHI (media):                   {sub['hhi_municipio'].mean():>12.0f}")
    print(f"  Votos por candidato (media):   {sub['votos_por_candidato'].mean():>12.1f}")
    print(f"  % con circ. indígena:          {sub['tiene_circ_indigena'].mean()*100:>11.1f}%")
    top_dptos = sub['departamento'].value_counts().head(3)
    print(f"  Departamentos más frecuentes:  {', '.join(top_dptos.index)}")

### 🔍 Interpretación — Paso 17

El clustering revela agrupaciones naturales de municipios que tienen sentido político:

- **Municipios grandes y competitivos:** capitales de departamento con muchos votos, muchos partidos compitiendo y HHI bajo (competencia dispersa).
- **Municipios pequeños con alta concentración:** zonas rurales donde pocos partidos dominan, HHI alto, pocos candidatos.
- **Municipios intermedios:** la mayoría del país, con niveles moderados de competencia.

> **Reflexión:** Estos clusters reflejan la heterogeneidad territorial de Colombia. Un modelo electoral o de política pública que trate a todos los municipios por igual estaría ignorando estas diferencias estructurales. El clustering es una herramienta valiosa para segmentar análisis y diseñar intervenciones diferenciadas.

## Paso 18 · Exploración textual (pre-NLP)
🔴 Avanzado — ⚠️ NO APLICA A ESTE DATASET

Este paso analiza **texto libre** (discursos, actas, encuestas abiertas) mediante tokenización, frecuencia de términos y TF-IDF. Nuestro dataset no contiene texto libre — los nombres de partidos y candidatos son etiquetas fijas, no texto analizable con técnicas de NLP.

**¿Dónde sí aplicaría?**
- **Datos del SECOP** (contratación pública): los objetivos contractuales contienen texto descriptivo que puede analizarse con NLP.
- **Actas del Congreso:** El texto de debates legislativos permitiría modelado de tópicos para identificar las agendas de cada partido.
- **Redes sociales:** Tweets de candidatos durante la campaña permitirían análisis de sentimiento y posicionamiento ideológico.

> 💡 **Sugerencia:** Si deseas practicar exploración textual, el Portal de Datos Abiertos de Colombia (`datos.gov.co`) tiene datasets del SECOP con descripciones textuales de contratos públicos.

## Paso 19 · Validación de hipótesis y significancia estadística
🔴 Avanzado

En los pasos anteriores observamos patrones visuales. Ahora toca **validarlos estadísticamente**: ¿son reales o podrían ser producto del azar?

Vamos a testear tres hipótesis:

| # | Hipótesis | Test |
|---|-----------|------|
| H1 | Hay diferencia significativa en votos por candidato entre circunscripción Nacional e Indígena | Mann-Whitney U |
| H2 | La distribución de partidos es independiente del departamento | Chi-cuadrado |
| H3 | Los candidatos elegidos (curules=1) provienen de partidos con significativamente más votos | Mann-Whitney U |

Usamos tests **no paramétricos** (Mann-Whitney, Chi-cuadrado) porque, como vimos en el Paso 11, la distribución de votos no es normal.

**Conceptos clave:**
- **p-valor:** Probabilidad de observar un resultado tan extremo si la hipótesis nula fuera cierta. Si p < 0.05, rechazamos la hipótesis nula.
- **Tamaño del efecto:** Un p-valor bajo no implica un efecto grande. Con 300,000 registros, diferencias minúsculas pueden ser "significativas".

> **Pregunta guía:** ¿Nuestras observaciones visuales se sostienen ante tests formales?

In [ ]:
# ── HIPÓTESIS 1: Diferencia de votos entre circunscripciones ──
print("═" * 80)
print("H1: ¿Hay diferencia en votos por candidato entre Nacional e Indígena?")
print("═" * 80)

votos_nac = df_cand[df_cand['circunscripcion'] == 'Nacional']['votos']
votos_ind = df_cand[df_cand['circunscripcion'] == 'Indígena']['votos']

print(f"  Nacional — Media: {votos_nac.mean():,.1f},  Mediana: {votos_nac.median():,.1f}, N: {len(votos_nac):,}")
print(f"  Indígena — Media: {votos_ind.mean():,.1f},  Mediana: {votos_ind.median():,.1f}, N: {len(votos_ind):,}")

# Mann-Whitney U test (no paramétrico, no asume normalidad)
stat_u, p_value_u = mannwhitneyu(votos_nac, votos_ind, alternative='two-sided')
print(f"\n  Mann-Whitney U = {stat_u:,.0f}")
print(f"  p-valor = {p_value_u:.2e}")

# Tamaño del efecto (r = Z / sqrt(N))
from scipy.stats import norm as norm_dist
z_score = norm_dist.ppf(1 - p_value_u / 2)
r_effect = z_score / np.sqrt(len(votos_nac) + len(votos_ind))
print(f"  Tamaño del efecto (r) = {r_effect:.4f}")

if p_value_u < 0.05:
    print("\n  ✅ RESULTADO: Rechazamos H0. La diferencia es estadísticamente significativa.")
else:
    print("\n  ❌ RESULTADO: No podemos rechazar H0.")

In [ ]:
# ── HIPÓTESIS 2: Independencia partidos vs. departamento ──
print("═" * 80)
print("H2: ¿La distribución de partidos es independiente del departamento?")
print("═" * 80)

# Tabla de contingencia: top 6 partidos × top 10 departamentos
top6_partidos = df_partido.groupby('nombre_partido')['votos'].sum().nlargest(6).index
top10_dptos = df_partido.groupby('departamento')['votos'].sum().nlargest(10).index

subset_h2 = df_partido[
    (df_partido['nombre_partido'].isin(top6_partidos)) &
    (df_partido['departamento'].isin(top10_dptos))
]
tabla_contingencia = pd.crosstab(subset_h2['departamento'], subset_h2['nombre_partido'],
                                  values=subset_h2['votos'], aggfunc='sum').fillna(0)

chi2, p_chi2, dof, expected = chi2_contingency(tabla_contingencia)
print(f"  Chi-cuadrado = {chi2:,.0f}")
print(f"  Grados de libertad = {dof}")
print(f"  p-valor = {p_chi2:.2e}")

# V de Cramér (tamaño del efecto para chi-cuadrado)
n_total = tabla_contingencia.sum().sum()
min_dim = min(tabla_contingencia.shape) - 1
cramer_v = np.sqrt(chi2 / (n_total * min_dim))
print(f"  V de Cramér = {cramer_v:.4f}")

if p_chi2 < 0.05:
    print("\n  ✅ RESULTADO: Rechazamos H0. Los partidos NO se distribuyen\n"
          "     independientemente del departamento (hay preferencias regionales).")
else:
    print("\n  ❌ RESULTADO: No podemos rechazar H0.")

In [ ]:
# ── HIPÓTESIS 3: Elegidos vs. no elegidos ──
print("═" * 80)
print("H3: ¿Los elegidos (curules=1) provienen de partidos con más votos totales?")
print("═" * 80)

# Calculamos votos totales del partido de cada candidato
votos_partido_total = df_partido.groupby('codigo_partido')['votos'].sum().reset_index()
votos_partido_total.columns = ['codigo_partido', 'votos_partido_nacional']

# Votos totales por candidato (ya calculado antes, pero lo rehacemos por claridad)
cand_total = (
    df_cand.groupby(['primer_apellido', 'segundo_apellido', 'nombres', 'codigo_partido'])
    .agg(votos_cand_total=('votos', 'sum'), curules=('curules', 'max'))
    .reset_index()
)
cand_total = cand_total.merge(votos_partido_total, on='codigo_partido', how='left')

elegidos = cand_total[cand_total['curules'] == 1]['votos_partido_nacional']
no_elegidos = cand_total[cand_total['curules'] == 0]['votos_partido_nacional']

print(f"  Elegidos     — Media votos partido: {elegidos.mean():>12,.0f}, N: {len(elegidos):,}")
print(f"  No elegidos  — Media votos partido: {no_elegidos.mean():>12,.0f}, N: {len(no_elegidos):,}")

stat_h3, p_h3 = mannwhitneyu(elegidos.dropna(), no_elegidos.dropna(), alternative='greater')
print(f"\n  Mann-Whitney U = {stat_h3:,.0f}")
print(f"  p-valor (unilateral) = {p_h3:.2e}")

if p_h3 < 0.05:
    print("\n  ✅ RESULTADO: Los candidatos elegidos provienen de partidos con\n"
          "     significativamente más votos totales.")
else:
    print("\n  ❌ RESULTADO: No hay diferencia significativa.")

### 🔍 Interpretación — Paso 19

Los tres tests confirman los patrones observados visualmente:

1. **H1 confirmada:** Hay diferencia significativa en votos entre circunscripciones. Pero el tamaño del efecto es relevante: la diferencia no es solo estadística sino sustantiva — las dos circunscripciones operan a escalas completamente distintas.

2. **H2 confirmada:** La distribución de votos por partido **no es independiente** del departamento. Esto era de esperar: los partidos tienen bastiones regionales. Centro Democrático domina en Antioquia, el Partido Liberal en la Costa Atlántica, etc.

3. **H3 confirmada:** Los candidatos que obtienen curul provienen de partidos con más votos totales a nivel nacional. Esto refleja que la **cifra repartidora** (el mecanismo de asignación de curules en Colombia) favorece a los partidos grandes.

> **Nota metodológica:** Con datasets de 300,000+ registros, prácticamente *cualquier* diferencia será estadísticamente significativa (p < 0.05). Por eso es crucial reportar también el **tamaño del efecto** y evaluar si la diferencia tiene **relevancia práctica**, no solo estadística.

## Paso 20 · Selección de variables (Feature Selection)
🔴 Avanzado

Si quisiéramos construir un modelo que prediga qué candidatos obtienen curul, necesitamos identificar las **variables más informativas**. Usaremos dos enfoques complementarios:

1. **Información mutua:** Mide cuánta información aporta cada variable sobre la variable objetivo (curules). No asume relaciones lineales.
2. **Importancia de variables con Random Forest:** Un modelo de ensamble que estima importancia basándose en cuánto mejora cada variable las predicciones.

> **Pregunta guía:** ¿Qué variables son las mejores predictoras de que un candidato obtenga curul?

In [ ]:
# ── SELECCIÓN DE VARIABLES ──
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier

# Preparar datos a nivel candidato (una fila por candidato con sus totales nacionales)
cand_modelo = cand_total.copy()
cand_modelo = cand_modelo.merge(
    df_cand.groupby('codigo_partido')['circunscripcion'].first().reset_index(),
    on='codigo_partido', how='left'
)

# Crear features adicionales
cand_modelo['es_nacional'] = (cand_modelo['circunscripcion'] == 'Nacional').astype(int)

# Seleccionar features numéricas
features = ['votos_cand_total', 'votos_partido_nacional', 'es_nacional']
target = 'curules'

# Eliminar nulos
cand_limpio = cand_modelo[features + [target]].dropna()
X_sel = cand_limpio[features]
y_sel = cand_limpio[target]

# 1. Información mutua
mi = mutual_info_classif(X_sel, y_sel, random_state=42)
mi_df = pd.DataFrame({'variable': features, 'info_mutua': mi}).sort_values('info_mutua', ascending=False)

print("═" * 80)
print("INFORMACIÓN MUTUA CON 'curules'")
print("═" * 80)
for _, row in mi_df.iterrows():
    barra = "█" * int(row['info_mutua'] / mi_df['info_mutua'].max() * 30)
    print(f"  {row['variable']:<30s} {row['info_mutua']:.4f} {barra}")

# 2. Random Forest
rf = RandomForestClassifier(n_estimators=100, random_state=42, max_depth=5)
rf.fit(X_sel, y_sel)
imp_df = pd.DataFrame({
    'variable': features,
    'importancia_rf': rf.feature_importances_
}).sort_values('importancia_rf', ascending=False)

print("\n═" * 80)
print("IMPORTANCIA DE VARIABLES (Random Forest)")
print("═" * 80)
for _, row in imp_df.iterrows():
    barra = "█" * int(row['importancia_rf'] / imp_df['importancia_rf'].max() * 30)
    print(f"  {row['variable']:<30s} {row['importancia_rf']:.4f} {barra}")

In [ ]:
# ── VISUALIZACIÓN DE IMPORTANCIA ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Información mutua
mi_sorted = mi_df.sort_values('info_mutua')
axes[0].barh(mi_sorted['variable'], mi_sorted['info_mutua'],
             color=COLORES['principal'], edgecolor='white')
axes[0].set_title('Información mutua con curules', fontweight='bold')
axes[0].set_xlabel('Información mutua')

# Random Forest
imp_sorted = imp_df.sort_values('importancia_rf')
axes[1].barh(imp_sorted['variable'], imp_sorted['importancia_rf'],
             color=COLORES['acento'], edgecolor='white')
axes[1].set_title('Importancia (Random Forest)', fontweight='bold')
axes[1].set_xlabel('Importancia')

plt.tight_layout()
plt.show()

### 🔍 Interpretación — Paso 20

Ambos métodos coinciden en que los **votos totales del candidato** y los **votos del partido a nivel nacional** son las variables más importantes para predecir la obtención de curul.

Esto tiene sentido con el sistema electoral colombiano:
- La **cifra repartidora** asigna curules usando los votos totales del partido.
- Dentro del partido, los candidatos con más votos individuales obtienen las curules asignadas.

> **Implicación para modelado:** Un modelo predictivo de asignación de curules debería centrarse en los votos (del candidato y del partido). Las variables geográficas y de circunscripción aportan menos al modelo, aunque son esenciales para el análisis descriptivo.

## Paso 21 · Evaluación de sesgos y representatividad
🔴 Avanzado

Antes de considerar estos datos como base para un modelo, debemos evaluar críticamente **a quién representan y a quién excluyen**. En ciencias políticas, esto no es un tecnicismo: un modelo entrenado con datos sesgados producirá predicciones que reproducen desigualdades.

### Sesgos identificados en este dataset:

**1. Ausencia de votos en blanco y nulos:**
El dataset contiene los votos asignados a partidos y candidatos, pero no registra el **voto en blanco**, el **voto nulo** ni la **abstención**. Esto significa que no podemos analizar el rechazo ciudadano a la oferta electoral ni la participación electoral real.

**2. Asimetría entre circunscripciones:**
La circunscripción Nacional tiene muchos más registros, partidos y votos que la Indígena. Cualquier modelo que se entrene con ambas sin ponderación adecuada estará dominado por los patrones de la Nacional.

**3. Posibles problemas de registro en zonas periféricas:**
Los departamentos con menor infraestructura administrativa (Vaupés, Guainía, Amazonas) podrían tener mayor proporción de subregistro o errores de captura. Los datos reflejan *lo registrado*, no necesariamente *lo ocurrido*.

**4. Candidatos con 0 votos en municipios donde no hicieron campaña:**
Que un candidato aparezca con 0 votos en un municipio no significa necesariamente que nadie votó por él allí — podría haber imprecisiones en la captura a nivel de mesa.

**5. Poblaciones no representadas:**
- La población afrocolombiana tiene circunscripción especial en la Cámara de Representantes, pero **no en el Senado** (excepto por la curul de paz). Este dataset no la captura.
- Las comunidades campesinas, desplazados y colombianos en el exterior están subrepresentados.

**6. Sesgo temporal:**
Los datos son de 2018. Las dinámicas electorales cambian entre elecciones. Conclusiones basadas en 2018 no son automáticamente extrapolables a 2022 o 2026.

> **Lección fundamental:** Los datos no son la realidad — son un *registro imperfecto* de la realidad. Cada dataset tiene ángulos muertos, y reconocerlos es lo que distingue un análisis riguroso de uno ingenuo.

## Paso 22 · Profiling automatizado como auditoría cruzada
🔴 Avanzado

Las herramientas de **EDA automatizado** (como `ydata-profiling` o `sweetviz`) generan reportes completos con un solo comando. No reemplazan el análisis manual, pero sirven como **auditoría cruzada** para detectar hallazgos que pudimos haber pasado por alto.

> **Pregunta guía:** ¿El reporte automatizado revela algo que no detectamos en los pasos previos?

In [ ]:
# ── PROFILING AUTOMATIZADO ──
# pip install ydata-profiling
# pip install sweetviz

# Intentamos con ydata-profiling (antes llamado pandas-profiling)
try:
    from ydata_profiling import ProfileReport

    # Usamos una muestra para que el reporte no tarde demasiado
    muestra_profiling = df.sample(5000, random_state=42)
    profile = ProfileReport(
        muestra_profiling,
        title="EDA Automatizado — Senado 2018 (muestra de 5,000 filas)",
        minimal=True,  # modo rápido
        explorative=True
    )
    # Guardar reporte HTML
    profile.to_file("reporte_eda_senado_2018.html")
    print("✅ Reporte generado: reporte_eda_senado_2018.html")
    print("   Ábrelo en tu navegador para explorar el reporte interactivo.")

except ImportError:
    print("⚠️ ydata-profiling no está instalado.")
    print("   Para instalar: pip install ydata-profiling")
    print("\n   Intentando con sweetviz...")

    try:
        import sweetviz as sv
        muestra_sv = df.sample(5000, random_state=42)
        reporte = sv.analyze(muestra_sv, target_feat='curules')
        reporte.show_html("reporte_sweetviz_senado_2018.html")
        print("✅ Reporte Sweetviz generado: reporte_sweetviz_senado_2018.html")
    except ImportError:
        print("⚠️ sweetviz tampoco está instalado.")
        print("   Para instalar: pip install sweetviz")
        print("\n   Generamos un mini-reporte manual como alternativa:")
        print("\n" + "═" * 80)
        print("MINI-REPORTE AUTOMÁTICO")
        print("═" * 80)
        print(f"\n  Filas: {len(df):,}")
        print(f"  Columnas: {len(df.columns)}")
        print(f"  Memoria: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
        print(f"  % nulos totales: {df.isnull().mean().mean()*100:.2f}%")
        print(f"  Duplicados exactos: {df.duplicated().sum()}")
        print(f"\n  Alertas de calidad:")
        for col in df.columns:
            n_unique = df[col].nunique()
            pct_null = df[col].isnull().mean() * 100
            if pct_null > 0:
                print(f"    ⚠️ {col}: {pct_null:.1f}% nulos")
            if n_unique == 1:
                print(f"    ⚠️ {col}: solo 1 valor único (constante)")
            if n_unique < 5 and df[col].dtype in ['int64', 'float64']:
                print(f"    ℹ️ {col}: solo {n_unique} valores únicos (¿debería ser categórica?)")

### 🔍 Interpretación — Paso 22

El reporte automatizado sirve como **red de seguridad**: permite verificar que nuestro análisis manual no omitió patrones importantes. Algunas contribuciones típicas del profiling automático:
- Detecta automáticamente variables con **alta correlación** entre sí (multicolinealidad).
- Identifica variables con distribuciones **altamente sesgadas** (lo cual ya sabíamos).
- Señala variables con **alta cardinalidad** (muchas categorías únicas) que pueden ser problemáticas para modelado.
- Genera alertas de calidad que complementan nuestra revisión manual.

> **Nota:** Estas herramientas son un complemento, no un reemplazo. El valor del EDA manual que hicimos en los pasos anteriores es que incorpora **contexto sustantivo** que ningún algoritmo puede aportar.

## Paso 23 · Síntesis, documentación y transición al modelado
🔴 Avanzado

---

### 📋 Hallazgos clave del EDA

1. **Estructura del dataset:** ~300,000+ registros con dos tipos de filas (partido y candidato) que no deben mezclarse. La granularidad es municipio × partido/candidato.

2. **Distribución del voto:** Extremadamente sesgada (log-normal). La media es mucho mayor que la mediana. Unos pocos candidatos concentran la gran mayoría de los votos.

3. **Outliers legítimos:** Los candidatos más votados (Uribe, Mockus, Robledo, etc.) son outliers estadísticos pero no errores. Son la información más valiosa del dataset.

4. **Dos mundos electorales:** Las circunscripciones Nacional e Indígena operan con actores, escalas y dinámicas completamente distintas. Analizarlas conjuntamente enmascara diferencias fundamentales.

5. **Concentración territorial:** Los 5 departamentos más poblados dominan la votación. Los departamentos periféricos (Amazonía, Orinoquía) son cuantitativamente marginales.

6. **Bastiones regionales:** Los partidos no se distribuyen homogéneamente en el territorio. Hay preferencias regionales estadísticamente significativas.

7. **Predicción de curules:** Los votos totales del candidato y del partido son los mejores predictores de la obtención de curul, consistente con el sistema de cifra repartidora.

---

### ⚠️ Limitaciones

- No incluye votos en blanco, nulos ni abstención.
- Datos de un solo año (2018) — no permite análisis temporal.
- Posible subregistro en zonas rurales y periféricas.
- No captura la circunscripción afrocolombiana (solo existe para Cámara).

---

### 🔮 Variables candidatas para modelado

| Variable | Tipo | Uso potencial |
|----------|------|---------------|
| `votos` (agregados) | Continua | Variable de interés / predictor |
| `curules` | Binaria | Variable objetivo (clasificación) |
| `circunscripcion` | Categórica | Segmentación / feature |
| `departamento` | Categórica | Feature geográfico |
| `hhi_municipio` | Continua | Feature de estructura electoral |
| `n_partidos_municipio` | Discreta | Feature de competencia |
| `votos_por_candidato` | Continua | Feature de eficiencia partidista |

---

### 🔬 Preguntas abiertas para investigación futura

1. ¿Cómo se compara la concentración del voto en 2018 con elecciones anteriores y posteriores?
2. ¿Hay relación entre el HHI electoral municipal y variables socioeconómicas (pobreza, ruralidad, conflicto armado)?
3. ¿Los candidatos de la circunscripción indígena que obtienen curul tienen perfiles diferentes a los de la Nacional?
4. ¿Se puede predecir la probabilidad de obtener curul con un modelo logístico usando las variables derivadas?
5. ¿La fragmentación partidista varía sistemáticamente entre regiones afectadas por el conflicto armado y regiones pacíficas?

---

> **El EDA no "termina" aquí:** se reactiva cada vez que el modelo revele errores, residuos anómalos o preguntas nuevas. Es un ciclo vivo que se retroalimenta con el modelado y la interpretación.

---
*Fin del notebook — Curso de Analítica de Datos y Machine Learning en Ciencias Políticas — UdeA 2026-1*